# Volume 7, Chapter 9: Curves, Growth and Conformal Maps (Python companion)

Exercises first, solutions at the end. Point-in-polygon tests and the Jordan curve theorem, the Mellin analysis of probabilistic counting, the Schwarz–Christoffel map onto a square, and the convexity bound for zeta measured. Requirements: numpy, scipy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★★) CS application: is the point inside the polygon?** Every graphics and GIS library must decide whether a point lies inside a polygon. Implement two tests: the *crossing number* (count how often a horizontal ray from the point crosses the boundary; inside if odd) and the *winding number*, computed with integers only, by counting signed upward and downward crossings of the ray (this is the continuous argument of Lemma 9.1, sampled only where the angle passes through $0$). (a) On random simple polygons, check that the two tests always agree; why is this the Jordan curve theorem in action, with winding number $\pm1$ inside? (b) On a pentagram, find points where they disagree, and explain what each test computes there. (c) Compare their speed with a version that sums the angles $\arg\frac{v_{k+1}-z}{v_k-z}$ with `atan2`.

**P9.2 (★★) CS application: counting distinct visitors.** (a) Simulate the maximum level $H$ of Theorem 9.24 for Poisson numbers of items with mean $x=10^2,\dots,10^6$, and compare the sample means with $\log_2x+\frac\gamma{\ln2}-\frac12$. (b) Compute $\mathbb E[H]=\sum_{k\ge1}(1-e^{-x2^{-k}})$ exactly with 30 digits for $\log_2x$ running over one period, subtract the main terms, and plot the remaining wobble against $P(\log_2x)$ computed from its Fourier series. How many simulated samples would you need to see it? (c) Turn $H$ into a crude distinct-count estimator for a stream of hashed words, and measure its error; why do Flajolet and Martin average over many independent hash functions?

**P9.3 (★) The square map of Figure 9.2.** Using the series $f(z)=\sum_k\binom{-1/2}k\frac{z^{4k+1}}{4k+1}$, (a) check numerically that $f$ preserves the right angle between radii and circles inside the disc, (b) check that the image of the circle $|z|=0.999$ is close to a square, and (c) show that the corner is $e^{i\pi/4}\int_0^1\frac{\mathrm dr}{\sqrt{1-r^4}}=e^{i\pi/4}\frac{\Gamma(1/4)^2}{4\sqrt{2\pi}}$, half the lemniscate constant of Volume 4 (Exercise 5.5). Why does the truncated series converge so slowly at the corner?

**P9.4 (★) The convexity bound, measured.** (a) Check Lemma 9.14(a): plot $|\Gamma(\sigma+it)|\big/\bigl(\sqrt{2\pi}|t|^{\sigma-1/2}e^{-\pi|t|/2}\bigr)$ for $\sigma=-1,0,\frac12,2$. (b) Compute $\max|\zeta(\frac12+it)|$ over the ranges $[T,2T]$ for $T=2^5,\dots,2^{12}$, and compare its growth with $T^{1/4}$ (Theorem 9.15), $T^{1/6}$ (Hardy and Littlewood) and with the Lindelöf hypothesis.

## Solutions

### P9.1

In [ ]:
import numpy as np, math, random, time

def crossing_number(z, V):
    x, y = z
    inside = False
    n = len(V)
    for k in range(n):
        (x1, y1), (x2, y2) = V[k], V[(k + 1) % n]
        if (y1 > y) != (y2 > y):                       # the edge straddles the horizontal line
            xc = x1 + (y - y1) * (x2 - x1) / (y2 - y1)
            if xc > x:
                inside = not inside
    return int(inside)

def winding_number(z, V):
    """Signed crossings of the rightward ray: +1 upward with z on the left of the edge, -1 downward with z on the right."""
    x, y = z
    w = 0
    n = len(V)
    for k in range(n):
        (x1, y1), (x2, y2) = V[k], V[(k + 1) % n]
        side = (x2 - x1) * (y - y1) - (x - x1) * (y2 - y1)   # > 0 when z is left of the edge
        if y1 <= y < y2 and side > 0:
            w += 1
        elif y2 <= y < y1 and side < 0:
            w -= 1
    return w

def winding_by_angles(z, V):
    zc = complex(*z)
    W = [complex(*v) - zc for v in V]
    total = sum(math.atan2((W[(k + 1) % len(W)] / W[k]).imag, (W[(k + 1) % len(W)] / W[k]).real) for k in range(len(W)))
    return round(total / (2 * math.pi))

def random_simple_polygon(n, rng):
    """A star-shaped polygon: random radii at sorted random angles (simple by construction)."""
    th = np.sort(rng.uniform(0, 2 * np.pi, n))
    r = rng.uniform(0.3, 1.0, n)
    return list(zip(r * np.cos(th), r * np.sin(th)))

rng = np.random.default_rng(1)
agree, inside_w = 0, set()
for trial in range(300):
    V = random_simple_polygon(rng.integers(3, 40), rng)
    for _ in range(50):
        z = tuple(rng.uniform(-1.1, 1.1, 2))
        c, w = crossing_number(z, V), winding_number(z, V)
        agree += (c == abs(w))
        inside_w.add(w)
print("agreements:", agree, "of", 300 * 50, "  winding numbers seen:", sorted(inside_w))

star = [(math.cos(math.pi / 2 + 4 * math.pi * k / 5), math.sin(math.pi / 2 + 4 * math.pi * k / 5)) for k in range(5)]
for z in [(0.0, 0.0), (0.0, 0.5), (0.5, 0.0), (0.0, 0.95), (2.0, 0.0)]:
    print(z, " crossing:", crossing_number(z, star), " winding:", winding_number(z, star), " angles:", winding_by_angles(z, star))

V = random_simple_polygon(200, rng)
pts = [tuple(rng.uniform(-1, 1, 2)) for _ in range(2000)]
for f in (crossing_number, winding_number, winding_by_angles):
    t0 = time.perf_counter()
    for z in pts:
        f(z, V)
    print(f"{f.__name__:18s} {time.perf_counter() - t0:.3f} s")

On simple polygons the two tests agree at every one of the $15\,000$ points, and the winding numbers seen are only $0$ and $1$ (these polygons are traversed counterclockwise): inside means winding number $1$, outside $0$, and the crossing parity is the winding number mod $2$. That the inside is a single region where the winding number is $1$ is the Jordan curve theorem for polygons, the case Jordan thought obvious. On the pentagram the centre has winding number $2$ but an even crossing number, so the even–odd rule declares the central pentagon *outside*, while the "nonzero" rule of PostScript, SVG and every font renderer declares it inside; in the points of the star both give $1$. In pure Python all three tests take a fraction of a second for $2000$ points and a $200$-gon, the crossing test being slowest here because it divides at every straddling edge. The integer winding test is exact, needs no trigonometry and no rounding of an angle sum, and this is why libraries compute winding numbers by counting signed crossings: the continuous argument changes by a full turn only when it passes through the ray.

### P9.2

In [ ]:
import numpy as np, mpmath as mp, math
rng = np.random.default_rng(2)
g = 0.5772156649015329

def sample_H(x, size):
    N = rng.poisson(x, size)
    # the maximum of N geometric levels: P(H < k) = (1 - 2^-k)^N; sample by inversion
    U = rng.random(size)
    H = np.zeros(size)
    m = N > 0
    # H >= k iff max >= k; with N items, P(H < k) = (1-2^-k)^N, so H = smallest k with U < (1-2^-(k+1))^N ...
    H[m] = np.floor(-np.log2(1 - U[m] ** (1.0 / N[m])))
    return H

for x in [1e2, 1e3, 1e4, 1e5, 1e6]:
    Hs = sample_H(x, 200000)
    print(f"x = {x:8.0f}  sample mean {Hs.mean():.4f} +- {Hs.std() / math.sqrt(len(Hs)):.4f}   main terms {math.log2(x) + g / math.log(2) - 0.5:.4f}")

mp.mp.dps = 30
def EH(x):
    return mp.nsum(lambda k: 1 - mp.exp(-x * mp.power(2, -k)), [1, mp.inf])

def P(u):
    return -2 / mp.log(2) * mp.re(sum(mp.gamma(2j * mp.pi * k / mp.log(2)) * mp.exp(-2j * mp.pi * k * u) for k in range(1, 4)))

us = np.linspace(0, 1, 21)
for u in us[::4]:
    x = mp.power(2, 20 + u)
    exact = EH(x) - (mp.log(x, 2) + mp.euler / mp.log(2) - mp.mpf(1) / 2)
    print(f"u = {u:.2f}   E[H] - main terms = {float(exact): .3e}   P(u) = {float(P(u)): .3e}")
amp = 2 / math.log(2) * float(abs(mp.gamma(2j * mp.pi / mp.log(2))))
print("amplitude of the first harmonic:", amp)

# a crude distinct-count estimator: hash words, level = number of trailing zeros
import hashlib
def level(word, salt):
    h = int.from_bytes(hashlib.blake2b((salt + word).encode(), digest_size=8).digest(), 'big')
    return (h & -h).bit_length() - 1 if h else 64
words = [f"visitor{i}" for i in range(50000)]
stream = [random_word for random_word in rng.choice(words, 200000)]
true = len(set(stream))
for m in [1, 16, 256]:
    Hmax = []
    for j in range(m):
        Hmax.append(max(level(w, str(j)) for w in set(stream[: 20000])) if m > 16 else max(level(w, str(j)) for w in stream))
    est = 2 ** (np.mean(Hmax) - g / math.log(2) + 0.5)
    print(f"{m:3d} hash functions: estimate {est:9.0f}   (true {true if m <= 16 else len(set(stream[:20000]))})")

The simulated means match $\log_2x+\frac\gamma{\ln2}-\frac12$ to within about two standard errors, which are about $0.004$ with $200\,000$ samples. The exact values, computed with $30$ digits, differ from the main terms by about $10^{-6}$, and the difference follows $P(\log_2x)$ to all printed digits: an oscillation of amplitude $1.573\cdot10^{-6}$ in the first harmonic. Since $H$ has standard deviation about $1.9$, seeing it by simulation would take some $(1.9/10^{-6})^2\approx4\cdot10^{12}$ samples: the fluctuation is a theorem, not an observation. The estimator $2^{\bar H-0.83+0.5}$, with $\bar H$ the average of the maxima over $m$ hash functions, has an error of about $1.9/\sqrt m$ in $\log_2$, that is a factor $2^{1.9/\sqrt m}$: up to a factor $3.7$ for one hash (here it happened to land within $6\%$, by luck), about $1.4$ for $16$ (here $26\%$ too high), and $1.09$ for $256$ (here $2\%$). Averaging over many hash functions (or, as HyperLogLog does, splitting the stream into buckets with one hash) reduces the error like $1/\sqrt m$, and the constant $\frac\gamma{\ln2}-\frac12$ that corrects the bias is exactly what the Mellin analysis computes. (The estimator above with $m=256$ runs on the first $20\,000$ stream items to keep the notebook fast.)

### P9.3

In [ ]:
import numpy as np, math, cmath
from scipy.special import binom, gamma
K = 20000
k = np.arange(K)
coef = binom(-0.5, k) / (4 * k + 1)
def f(z):
    return np.sum(coef * z ** (4 * k + 1))
def fprime(z):
    return 1 / np.sqrt(1 + z ** 4)

# (a) angle between the images of a radius and a circle at a point z = r e^{it}
worst = 0
for r in [0.3, 0.6, 0.9, 0.99]:
    for t in np.linspace(0, 2 * np.pi, 13)[:-1]:
        z, h = r * cmath.exp(1j * t), 1e-6
        dr = (f(z * (1 + h)) - f(z)) / h           # derivative along the radius
        dt = (f(z * cmath.exp(1j * h)) - f(z)) / h # along the circle
        ang = abs(cmath.phase(dt / dr))
        worst = max(worst, abs(ang - math.pi / 2))
print("largest deviation from a right angle:", worst)

# (b) the image of |z| = 0.999 in the sup norm: a square has constant max(|Re|, |Im|) after rotation
ts = np.linspace(0, 2 * np.pi, 400, endpoint=False)
img = np.array([f(0.999 * cmath.exp(1j * t)) for t in ts])
side = np.maximum(abs(img.real), abs(img.imag))
print("half-side of the image of |z|=0.999: min %.4f, max %.4f" % (side.min(), side.max()))

# (c) the corner
corner = gamma(0.25) ** 2 / (4 * math.sqrt(2 * math.pi))
partial = [np.sum(np.abs(binom(-0.5, np.arange(n))) / (4 * np.arange(n) + 1)) for n in [100, 1000, 10000]]
print("int_0^1 dr/sqrt(1-r^4) =", corner, " partial sums of the series at the corner:", partial)
print("corner as a point:", corner * cmath.exp(1j * math.pi / 4))

Inside the disc the images of radii and circles cross at right angles to within the accuracy of the finite differences, as conformality demands. The image of $|z|=0.999$ already has half-side between about $0.905$ and $0.926$ in the sup norm: it is a square with slightly rounded corners, which sharpen as $r\to1$. The corner is $e^{i\pi/4}\cdot1.31103$, where $1.31103=\frac{\Gamma(1/4)^2}{4\sqrt{2\pi}}$ is half of the lemniscate constant $\varpi=2.62206$. At the corner the series is $\sum_k\frac{|\binom{-1/2}k|}{4k+1}$, with terms about $\frac{k^{-3/2}}{4\sqrt\pi}$, so the tail after $n$ terms is about $\frac1{2\sqrt{\pi n}}$: $10\,000$ terms still miss the third decimal. The map is smooth inside, but $f'(z)=(1+z^4)^{-1/2}$ blows up at the four points that become corners, which is exactly how a conformal map manufactures a right angle out of the straight angle of the circle.

### P9.4

In [ ]:
import mpmath as mp, numpy as np, math
mp.mp.dps = 15
for s in [-1, 0, 0.5, 2]:
    ratios = [float(abs(mp.gamma(s + 1j * t)) / (math.sqrt(2 * math.pi) * t ** (s - 0.5) * math.exp(-math.pi * t / 2))) for t in [1, 2, 5, 10, 50, 200]]
    print(f"sigma = {s:4}: ", " ".join(f"{r:.4f}" for r in ratios))

print()
rows = []
for j in range(5, 13):
    T = 2 ** j
    ts = np.linspace(T, 2 * T, 20 * T // 32 + 200)
    M = max(float(abs(mp.zeta(0.5 + 1j * t))) for t in ts)
    rows.append((T, M))
    print(f"T = {T:5d}  max|zeta| on [T,2T] = {M:7.3f}   T^(1/4) = {T ** 0.25:6.2f}   T^(1/6) = {T ** (1 / 6):6.2f}   log T = {math.log(T):5.2f}")
Ts, Ms = np.array(rows).T
print("fitted exponent:", np.polyfit(np.log(Ts), np.log(Ms), 1)[0])

The ratio for $\Gamma$ tends to $1$ on every line, quickly, as Lemma 9.14(a) says, with an error of order $\frac1t$. The maxima of $|\zeta(\frac12+it)|$ over dyadic ranges grow from about $4$ to about $16$, and over this short range the fitted exponent comes out near $\frac14$ ($0.28$ here): numerics at such low heights cannot tell a power from a slowly growing function. The maximum is conjectured (Farmer, Gonek and Hughes, 2007) to grow like $\exp\bigl((\frac1{\sqrt2}+o(1))\sqrt{\log T\log\log T}\bigr)$, smaller than every power of $T$, but this only shows at astronomically large heights, where $\log\log T$ is no longer tiny. This is a useful warning: asymptotic statements about $\zeta$, the convexity bound, the subconvexity records and the Lindelöf hypothesis alike, are about the limit, and experiments at $T\le10^4$ say little about them. Closing even part of the gap between $\frac14$ and $0$ takes arithmetic (exponential sums), not complex analysis alone. Our sampling grid can miss the exact peaks, so these maxima are slight underestimates.